# ML Lab Practical — AdaBoost Regressor

This notebook follows the established regression-practical pattern and the constraints discussed for the exam.

**Included:** CSV loading, EDA, missing-value handling, categorical encoding, ID removal, mandatory hyperparameter tuning, five outer folds with three-fold inner tuning, fold-wise and mean/std metrics, PCA comparison, visualizations, and a paired two-sided t-test.

**Target:** `Remaining_Useful_Life_Hours`. Change `TARGET` in the code if your dataset uses another target column.

**Leakage prevention:** imputation, encoding, scaling and PCA are inside the model pipeline; hyperparameters are tuned only on each outer training fold.

**PCA note:** PCA is included to match your previous notebooks. Tree-boosting models do not generally require scaling, and PCA may not improve them; report the measured results and follow your exact question paper.

**How to run:** Put the notebook and CSV in the same folder. One CSV is auto-selected; if multiple CSVs exist, enter the desired filename.

**Tuned hyperparameters:** `n_estimators`, `learning_rate`, `loss`, and base-tree `max_depth`.

**Estimator tuning clarification:** The base learner is a `DecisionTreeRegressor`. The grid tunes the estimator's tree depth, minimum split size, and minimum leaf size, along with AdaBoost's `n_estimators`, `learning_rate`, and `loss`. This treats estimator structure as part of hyperparameter tuning.


In [ ]:
import glob
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_rel
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
TARGET = "Remaining_Useful_Life_Hours"

# ---------- 1. LOAD CSV ----------
csv_files = glob.glob("*.csv")
if len(csv_files) == 1:
    file_name = csv_files[0]
elif len(csv_files) > 1:
    print("CSV files found:", csv_files)
    file_name = input("Enter the CSV filename to use: ").strip()
else:
    file_name = input("Enter the CSV file path: ").strip()

df = pd.read_csv(file_name)
print("Dataset shape:", df.shape)
display(df.head())
print("\\nDataset information:")
df.info()
print("\\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))
print("\\nDescriptive statistics:")
display(df.describe(include="all").T)
print("\\nDuplicate rows:", df.duplicated().sum())

# ---------- 2. SIMPLE EDA ----------
if TARGET not in df.columns:
    print("Available columns:", list(df.columns))
    raise ValueError("Change TARGET to the exact regression target column name.")

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.drop_duplicates().dropna(subset=[TARGET]).copy()

plt.figure(figsize=(7, 4))
sns.histplot(df[TARGET], kde=True)
plt.title("Target Distribution")
plt.xlabel(TARGET)
plt.tight_layout()
plt.show()

numeric_for_eda = df.select_dtypes(include=np.number).drop(columns=[TARGET], errors="ignore")
if len(numeric_for_eda.columns) > 0:
    numeric_for_eda.hist(figsize=(12, 8), bins=20)
    plt.suptitle("Numeric Feature Distributions")
    plt.tight_layout()
    plt.show()
    plt.figure(figsize=(9, 6))
    sns.heatmap(df.select_dtypes(include=np.number).corr(), cmap="coolwarm", center=0)
    plt.title("Numeric Feature Correlation")
    plt.tight_layout()
    plt.show()
    plt.figure(figsize=(12, 5))
    sns.boxplot(data=numeric_for_eda)
    plt.title("Numeric Feature Boxplots (Outlier Inspection)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

for col in df.select_dtypes(exclude=np.number).columns:
    print(f"\\n{col} value counts:")
    display(df[col].value_counts(dropna=False).head(10))

# ---------- 3. PREPARE X AND y ----------
id_cols = [
    c for c in df.columns
    if c != TARGET and (
        c.lower() in ["id", "record_id", "recordid", "machine_id"] or
        ("id" in c.lower() and df[c].nunique(dropna=False) >= 0.9 * len(df))
    )
]
print("Identifier columns removed:", id_cols)
X = df.drop(columns=[TARGET] + id_cols)
y = df[TARGET].astype(float)
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()
print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", encoder)
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
], remainder="drop")

pca_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE))
    ]), numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
], remainder="drop")

def mape_percent(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = np.abs(y_true) > 1e-8
    if not np.any(mask):
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


# ---------- 4. HYPERPARAMETER GRID ----------
# AdaBoostRegressor hyperparameters tuned:
# estimator: type of weak learner (DecisionTreeRegressor here).
# estimator__max_depth: complexity of each base tree.
# estimator__min_samples_split / estimator__min_samples_leaf: base-tree regularization.
# n_estimators: number of boosting stages.
# learning_rate: contribution of each stage.
# loss: loss function used to update weights.
#
# We use a DecisionTreeRegressor as the base estimator and tune its key structural
# parameters as well as AdaBoost's own parameters.
param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__learning_rate": [0.01, 0.1, 1.0],
    "model__loss": ["linear", "square", "exponential"],
    "model__estimator__max_depth": [1, 2, 3],
    "model__estimator__min_samples_split": [2, 5],
    "model__estimator__min_samples_leaf": [1, 2]
}
n_combinations = (
    len(param_grid["model__n_estimators"]) *
    len(param_grid["model__learning_rate"]) *
    len(param_grid["model__loss"]) *
    len(param_grid["model__estimator__max_depth"]) *
    len(param_grid["model__estimator__min_samples_split"]) *
    len(param_grid["model__estimator__min_samples_leaf"])
)
print("Parameter combinations per search:", n_combinations)
# ---------- 5. NESTED EVALUATION: SAME 5 OUTER FOLDS ----------
outer_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []
fold_predictions = {}

for fold, (train_idx, valid_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    for name, prep in [("Without PCA", preprocessor), ("With PCA", pca_preprocessor)]:
        pipe = Pipeline([
            ("preprocess", prep),
            ("model", AdaBoostRegressor(estimator=DecisionTreeRegressor(random_state=RANDOM_STATE), random_state=RANDOM_STATE))
        ])
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            scoring="neg_root_mean_squared_error",
            cv=3,
            n_jobs=-1,
            refit=True
        )
        start = time.time()
        grid.fit(X_train, y_train)
        elapsed = time.time() - start

        best_model = grid.best_estimator_
        train_pred = best_model.predict(X_train)
        valid_pred = best_model.predict(X_valid)
        train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
        valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
        valid_mae = mean_absolute_error(y_valid, valid_pred)
        valid_r2 = r2_score(y_valid, valid_pred)
        valid_mape = mape_percent(y_valid, valid_pred)

        results.append({
            "Fold": fold, "Model": name,
            "Train_RMSE": train_rmse, "Validation_RMSE": valid_rmse,
            "MAE": valid_mae, "R2": valid_r2, "MAPE_percent": valid_mape,
            "Train_Validation_RMSE_Gap": valid_rmse - train_rmse,
            "Best_Inner_CV_RMSE": -grid.best_score_,
            "Best_Params": grid.best_params_, "Runtime_seconds": elapsed
        })
        fold_predictions[(name, fold)] = (y_valid.to_numpy(), valid_pred)

        print(f"\\nFold {fold} | {name}")
        print("Best parameters:", grid.best_params_)
        print("Best inner CV RMSE:", round(-grid.best_score_, 4))
        print("Train RMSE:", round(train_rmse, 4))
        print("Validation RMSE:", round(valid_rmse, 4))
        print("Validation MAE:", round(valid_mae, 4))
        print("Validation R²:", round(valid_r2, 4))
        print("Validation MAPE (%):", round(valid_mape, 4))
        print("Runtime (seconds):", round(elapsed, 2))

results_df = pd.DataFrame(results)
print("\\nFOLD-WISE RESULTS")
display(results_df[[
    "Fold", "Model", "Train_RMSE", "Validation_RMSE", "MAE", "R2",
    "MAPE_percent", "Train_Validation_RMSE_Gap", "Best_Inner_CV_RMSE",
    "Runtime_seconds"
]].round(4))
print("\\nBEST HYPERPARAMETERS FOR EVERY FOLD")
display(results_df[["Fold", "Model", "Best_Params"]])

metric_cols = ["Validation_RMSE", "MAE", "R2", "MAPE_percent",
               "Train_Validation_RMSE_Gap", "Runtime_seconds"]
summary = results_df.groupby("Model")[metric_cols].agg(["mean", "std"]).round(4)
print("\\nMEAN AND STANDARD DEVIATION ACROSS THE 5 OUTER FOLDS")
display(summary)

# ---------- 6. PCA VARIANCE PLOT ----------
if len(numeric_cols) >= 1:
    eda_numeric = SimpleImputer(strategy="median").fit_transform(X[numeric_cols])
    eda_scaled = StandardScaler().fit_transform(eda_numeric)
    pca_eda = PCA().fit(eda_scaled)
    cumulative = np.cumsum(pca_eda.explained_variance_ratio_)
    plt.figure(figsize=(7, 4))
    plt.plot(range(1, len(cumulative) + 1), cumulative, marker="o")
    plt.axhline(0.95, linestyle="--", label="95% variance")
    plt.xlabel("Number of Principal Components")
    plt.ylabel("Cumulative Explained Variance")
    plt.title("PCA Cumulative Explained Variance")
    plt.legend()
    plt.tight_layout()
    plt.show()
    print("Components needed for at least 95% variance:", np.argmax(cumulative >= 0.95) + 1)

# ---------- 7. RMSE COMPARISON ----------
pivot_rmse = results_df.pivot(index="Fold", columns="Model", values="Validation_RMSE")
display(pivot_rmse.round(4))
plt.figure(figsize=(7, 4))
for col in pivot_rmse.columns:
    plt.plot(pivot_rmse.index, pivot_rmse[col], marker="o", label=col)
plt.xlabel("Outer Fold")
plt.ylabel("Validation RMSE (lower is better)")
plt.title("AdaBoost Regressor: With vs Without PCA")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(data=results_df, x="Model", y="Validation_RMSE")
sns.stripplot(data=results_df, x="Model", y="Validation_RMSE", color="black", alpha=0.6)
plt.title("Fold-wise Validation RMSE Distribution")
plt.tight_layout()
plt.show()

# ---------- 8. PAIRED TWO-SIDED T-TEST ----------
no_pca = pivot_rmse["Without PCA"].to_numpy()
with_pca = pivot_rmse["With PCA"].to_numpy()
t_stat, p_value = ttest_rel(with_pca, no_pca)
alpha = 0.05
print("\\nPAIRED TWO-SIDED T-TEST")
print("H0: Mean paired RMSE difference (With PCA - Without PCA) = 0")
print("H1: Mean paired RMSE difference (With PCA - Without PCA) != 0")
print("Alpha:", alpha)
print("t-statistic:", round(t_stat, 5))
print("p-value:", round(p_value, 6))
if p_value < alpha:
    print("Decision: Reject H0; difference is statistically significant at 5%.")
else:
    print("Decision: Fail to reject H0; insufficient evidence of a significant difference at 5%.")

# ---------- 9. ACTUAL VS PREDICTED AND RESIDUAL PLOTS ----------
for name in ["Without PCA", "With PCA"]:
    actual, predicted = fold_predictions[(name, 5)]
    residuals = actual - predicted
    plt.figure(figsize=(5, 5))
    plt.scatter(actual, predicted, alpha=0.7)
    low, high = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
    plt.plot([low, high], [low, high], linestyle="--")
    plt.xlabel("Actual target")
    plt.ylabel("Predicted target")
    plt.title(f"Actual vs Predicted — {name} (Fold 5)")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.scatter(predicted, residuals, alpha=0.7)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted target")
    plt.ylabel("Residual (Actual - Predicted)")
    plt.title(f"Residual Plot — {name} (Fold 5)")
    plt.tight_layout()
    plt.show()

# ---------- 10. CONCLUSION GUIDE ----------
mean_rmse = results_df.groupby("Model")["Validation_RMSE"].mean()
print("\\nCONCLUSION GUIDE")
print("Lower mean validation RMSE is better. Best by mean RMSE:", mean_rmse.idxmin())
print("Compare MAE, R², train-validation gap and runtime.")
print("AdaBoost sequentially focuses on observations that previous weak learners predicted poorly.")


# VERSION / INSTALL NOTE
print('If your scikit-learn is older and `estimator=` is rejected, replace `estimator=` with `base_estimator=` in AdaBoostRegressor.')

# ---------- 11. FEATURE IMPORTANCE (FINAL FULL-DATA FIT) ----------
# This is descriptive; report outer-fold metrics above as performance estimates.
final_pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", AdaBoostRegressor(estimator=DecisionTreeRegressor(random_state=RANDOM_STATE), random_state=RANDOM_STATE))
])
final_grid = GridSearchCV(final_pipe, param_grid=param_grid,
                          scoring="neg_root_mean_squared_error", cv=5,
                          n_jobs=-1, refit=True)
final_grid.fit(X, y)
print("Final full-data best parameters:", final_grid.best_params_)
print("Full-data CV RMSE:", round(-final_grid.best_score_, 4))
best_final = final_grid.best_estimator_
feature_names = best_final.named_steps["preprocess"].get_feature_names_out()
importances = best_final.named_steps["model"].feature_importances_
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df = importance_df.sort_values("Importance", ascending=False).head(15)
display(importance_df)
plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Top Feature Importances")
plt.tight_layout()
plt.show()
